In [ ]:
import os

# create the configuration for a run that will create multiple chunk-snaps in the logs.
# Try to create a 'new interval' situation, too.


# ---------- init a cluster with some data
#  (so that we can use manual commands)

# List the current directory
#print(os.listdir(.))

In [ ]:
# Ask the user to select a digit in the range 0 to 8
k = int(input("Please select a digit in the range 0 to 8: "))

# Check if the digit is within the valid range
if 0 <= k <= 8:
    print(f"will be running in k{k}")
else:
    print("Invalid selection. Please select a digit between 0 and 8.")
    exit(1)

from pathlib import Path

wd = Path(f"/home/rfriedma/src/k{k}/ceph/build")
os.chdir(wd)
%env CEPH_JTEST_ROOT=/home/rfriedma/src/k{k}/ceph/build
!echo $CEPH_JTEST_ROOT > /tmp/jpath


In [ ]:
!ls -l
!bash -c MGR=0 ls -l


In [ ]:
%%bash
scrtch=to_"`date +%d_%H%M`"
echo $scrtch

MDS=0 MGR=1 OSD=4 MON=1 ../src/vstart.sh -n  --without-dashboard --msgr2 -X --memstore -o "memstore_device_bytes=68435456" -o "osd_op_queue=wpq"
sleep 2
bin/ceph -s

#bin/ceph tell osd.* config set debug_osd 20/20

bin/ceph config set global osd_pool_default_pg_autoscale_mode off
sleep 2

# disable rescheduling of the queue due to 'no-scrub' flags
#bin/ceph tell osd.* config set osd_scrub_backoff_ratio 0.9999

# initial set of global scrub scheduling parameters
bin/ceph tell osd.* config set osd_scrub_interval_randomize_ratio 0.0
bin/ceph tell osd.* config set osd_deep_scrub_randomize_ratio 0
bin/ceph tell osd.* config set osd_scrub_min_interval 10
bin/ceph tell osd.* config set osd_scrub_max_interval 2000
bin/ceph tell osd.* config set osd_deep_scrub_interval 600


#PL1 is of size 3

bin/ceph osd pool create pl1 4 4
#bin/ceph osd pool autoscale-status
sleep 1
pl1_num=`bin/ceph osd pool stats pl1 | sed -n -r -e 's/.*id[^0-9]*([0-9]+)$/\1/p'`
echo $pl1_num
bin/ceph osd pool set pl1 size 4
bin/ceph osd pool set pl1 min_size 3
bin/ceph osd pool set pl1 pg_autoscale_mode off
bin/ceph osd pool stats
bin/ceph osd pool set pl1 noscrub 1
bin/ceph osd pool set pl1 nodeep-scrub 1
sleep 2

bin/rados bench -p pl1 -t 1 1 write -b 4096 --max-objects 8  --no-cleanup; 
bin/rados bench -p pl1 1 write -b 4096 --max-objects 128 --show-time --no-cleanup --run-name eeeee

#PL2

# bin/ceph osd pool create pl2 8 8
# bin/ceph osd pool set pl2 size 3
# bin/ceph osd pool set pl2 min_size 3
# bin/ceph osd pool set pl2 pg_autoscale_mode off
# bin/ceph osd pool stats
# bin/ceph osd pool set pl2 noscrub 0
# bin/ceph osd pool set pl2 nodeep-scrub 0
# sleep 2
# 
# bin/rados bench -p pl2 -t 1 1 write -b 4096 --max-objects 8  --no-cleanup; 
# bin/rados bench -p pl2 1 write -b 4096 --max-objects 128 --show-time --no-cleanup --run-name eeeee
# sleep 1

bin/ceph tell osd.* config set debug_osd 10/10
bin/ceph tell osd.0 dump_scrubs --format=json-pretty > /tmp/ds_00.json
bin/ceph tell osd.1 dump_scrubs --format=json-pretty > /tmp/ds_01.json
bin/ceph tell osd.2 dump_scrubs --format=json-pretty > /tmp/ds_02.json
bin/ceph tell osd.3 dump_scrubs --format=json-pretty > /tmp/ds_03.json


In [ ]:
%%bash
# set scrub parameters to guarantee slow scrub
bin/ceph tell osd.* config set osd_scrub_sleep "1.0"
bin/ceph tell osd.* config set osd_max_scrubs 2
bin/ceph tell osd.* config set osd_scrub_chunk_min 2
bin/ceph tell osd.* config set osd_shallow_scrub_chunk_min 2
bin/ceph tell osd.* config set osd_scrub_chunk_max 3
bin/ceph tell osd.* config set osd_shallow_scrub_chunk_max 3

bin/ceph tell osd.* config set debug_osd 20/20
bin/ceph osd pool set pl1 noscrub 0
bin/ceph osd pool set pl1 nodeep-scrub 0
sleep 2
pl1_num=`bin/ceph osd pool stats pl1 | sed -n -r -e 's/.*id[^0-9]*([0-9]+)$/\1/p'`
echo $pl1_num

bin/ceph tell $pl1_num.0 schedule-scrub
bin/ceph tell $pl1_num.1 schedule-scrub
bin/ceph tell $pl1_num.2 schedule-scrub
bin/ceph tell $pl1_num.2 schedule-deepscrub



In [ ]:
%%bash
# try to abort a running scrub with 'new interval'
pl1_num=`bin/ceph osd pool stats pl1 | sed -n -r -e 's/.*id[^0-9]*([0-9]+)$/\1/p'`
echo $pl1_num




In [ ]:
%%bash

pl1_num=`bin/ceph osd pool stats pl1 | sed -n -r -e 's/.*id[^0-9]*([0-9]+)$/\1/p'`
echo $pl1_num
t7=$pl1_num.7
t6=$pl1_num.7

# list the scrub queue
scrtch=to_"`date +'%H%M%S'`"
echo $scrtch

bin/ceph pg $t7 query -f json-pretty > /tmp/q7_a$scrtch
bin/ceph pg $t6 query -f json-pretty > /tmp/q7_a$scrtch
bin/ceph tell osd.0 dump_scrubs --format=json-pretty
bin/ceph tell osd.1 dump_scrubs --format=json-pretty
bin/ceph tell osd.2 dump_scrubs --format=json-pretty
bin/ceph tell osd.0 dump_scrubs --format=json-pretty > /tmp/dsB_0$scrtch.json
bin/ceph tell osd.1 dump_scrubs --format=json-pretty > /tmp/dsB_1$scrtch.json
bin/ceph tell osd.2 dump_scrubs --format=json-pretty > /tmp/dsB_2$scrtch.json

bin/ceph tell $t7 deep-scrub
bin/ceph tell $t7 scrub
bin/ceph pg $t7 query -f json-pretty > /tmp/q7_b$scrtch
sleep 0.5
bin/ceph pg $t7 query -f json-pretty >> /tmp/q7_b$scrtch
sleep 1.5
bin/ceph pg $t7 query -f json-pretty >> /tmp/q7_b$scrtch
bin/ceph pg $t6 query -f json-pretty >> /tmp/q6_b$scrtch
bin/ceph tell osd.0 dump_scrubs --format=json-pretty > /tmp/dsC_0$scrtch.json
bin/ceph tell osd.1 dump_scrubs --format=json-pretty > /tmp/dsC_1$scrtch.json
bin/ceph tell osd.2 dump_scrubs --format=json-pretty > /tmp/dsC_2$scrtch.json






In [ ]:
%%bash
echo '-------->' $CEPH_JTEST_ROOT
jp=$CEPH_JTEST_ROOT
if [ -z $CEPH_JTEST_ROOT ]; then
    echo "CEPH_JTEST_ROOT is not set"
    [[ -f /tmp/jpath ]] && jp=`cat /tmp/jpath` || jp='.'
    echo '-------->' $jp
    %env CEPH_JTEST_ROOT=$jp
fi
cd $jp


In [ ]:
%%bash

file_path="/tmp/jpath"
if [ -f "$file_path" ]; then
        file_contents=$(cat "$file_path")
        echo "File contents read into variable."
else
        echo "File does not exist."
fi

In [ ]:
%%bash

#cd $CEPH_JTEST_ROOT

bin/ceph tell osd.0 dump_scrubs --format=json-pretty > /tmp/ds_00_b4params.json
bin/ceph tell osd.1 dump_scrubs --format=json-pretty > /tmp/ds_01_b4params.json
bin/ceph tell osd.2 dump_scrubs --format=json-pretty > /tmp/ds_02_b4params.json


# set the scheduling parameters

bin/ceph osd pool set pl1 scrub_min_interval 1
bin/ceph osd pool set pl1 scrub_max_interval 2
bin/ceph osd pool set pl1 deep_scrub_interval 3

#bin/ceph osd pool set pl2 scrub_min_interval 2
#bin/ceph osd pool set pl2 scrub_max_interval 4
#bin/ceph osd pool set pl2 deep_scrub_interval 10

sleep 3

bin/ceph tell osd.0 dump_scrubs --format=json-pretty > /tmp/ds_00.json
bin/ceph tell osd.1 dump_scrubs --format=json-pretty > /tmp/ds_01.json
bin/ceph tell osd.2 dump_scrubs --format=json-pretty > /tmp/ds_02.json





In [ ]:
%%bash

# list the scrub queue
scrtch=to_"`date +'%H%M%S'`"
echo $scrtch
bin/ceph tell osd.0 dump_scrubs --format=json-pretty
bin/ceph tell osd.1 dump_scrubs --format=json-pretty
bin/ceph tell osd.2 dump_scrubs --format=json-pretty
bin/ceph tell osd.0 dump_scrubs --format=json-pretty > /tmp/ds_0$scrtch.json
bin/ceph tell osd.1 dump_scrubs --format=json-pretty > /tmp/ds_1$scrtch.json
bin/ceph tell osd.2 dump_scrubs --format=json-pretty > /tmp/ds_2$scrtch.json


In [ ]:
%%bash

pl1_num=`bin/ceph osd pool stats pl1 | sed -n -r -e 's/.*id[^0-9]*([0-9]+)$/\1/p'`
echo $pl1_num
bin/ceph tell osd.* config set osd_scrub_sleep "3.0"
bin/ceph tell osd.* config set osd_max_scrubs 1
bin/ceph tell osd.* config set osd_scrub_chunk_max 5
bin/ceph tell osd.* config set osd_shallow_scrub_chunk_max 5

bin/ceph tell osd.* config set osd_stats_update_period_scrubbing 2
bin/ceph tell osd.* config set osd_stats_update_period_not_scrubbing 2
#bin/ceph tell mgr.$(bin/ceph mgr services | jq -r .mgr) config set mgr_stats_period 2
sleep 1

# set higher urgency to one of the PGs
bin/ceph tell $pl1_num.7 scrub
bin/ceph tell $pl1_num.3 deep-scrub
bin/ceph tell $pl1_num.1 deep-scrub
bin/ceph tell $pl1_num.6 schedule-deep-scrub
echo '---------------'
bin/ceph tell osd.0 dump_scrubs --format=json-pretty
echo '---------------'
bin/ceph tell osd.1 dump_scrubs --format=json-pretty
echo '---------------'
bin/ceph tell osd.2 dump_scrubs --format=json-pretty
sleep 1
bin/ceph pg dump pgs
echo '---------------'
#bin/ceph tell osd.0 dump_scrubs --format=json-pretty
echo '---------------'
bin/ceph tell osd.1 dump_scrubs --format=json-pretty
echo '---------------'
#bin/ceph tell osd.2 dump_scrubs --format=json-pretty
sleep 2
bin/ceph pg dump pgs
echo '---------------'
bin/ceph tell osd.0 dump_scrubs --format=json-pretty
echo '---------------'
bin/ceph tell osd.1 dump_scrubs --format=json-pretty
echo '---------------'
bin/ceph tell osd.2 dump_scrubs --format=json-pretty
sleep 2
bin/ceph pg dump pgs
echo '---------------'
#bin/ceph tell osd.0 dump_scrubs --format=json-pretty
echo '---------------'
bin/ceph tell osd.1 dump_scrubs --format=json-pretty
echo '---------------'
#bin/ceph tell osd.2 dump_scrubs --format=json-pretty



In [ ]:
%%bash

scrtch=to_"`date +'%H%M%S'`"
echo $scrtch

# list the scrub queue
bin/ceph tell osd.0 dump_scrubs --format=json-pretty > /tmp/ds_0$scrtch.json
bin/ceph tell osd.1 dump_scrubs --format=json-pretty > /tmp/ds_1$scrtch.json
bin/ceph tell osd.2 dump_scrubs --format=json-pretty > /tmp/ds_2$scrtch.json
jq -s 'add' /tmp/ds_0$scrtch.json /tmp/ds_1$scrtch.json /tmp/ds_2$scrtch.json > /tmp/ds_all$scrtch.json
bin/ceph tell osd.0 dump_scrubs --format=json-pretty
bin/ceph tell osd.1 dump_scrubs --format=json-pretty
bin/ceph tell osd.2 dump_scrubs --format=json-pretty


In [ ]:
raise SystemExit("Stop here")

In [ ]:
%%bash

#cat /tmp/ds_01.json | jq -s '[.[]|.[]|select(.eligible==true)]| sort_by(.overdue,.pgid)' |  head -20

#cat /tmp/ds_01.json | jq -s '[.[]|.[]|select(.eligible==true)|  { pgid, overdue, sched_time } ]' |  head -20

#cat /tmp/ds_01.json | jq -s '[.[]|.[]|select(.eligible==true)|  { pgid, overdue, sched_time } ]' |  head -20

#cat ds_01.json | jq -s '[.[]|.[]|select(.eligible==false) | .overdue as $ov | . += { "ov":$ov } | [.] ] '|  head -20

#cat ds_01.json | jq -s '[.[]|.[]|select(.eligible==false) | .overdue as $ov | . += { "ov":$ov|not } | [.] ] '|  head -20

cat /tmp/ds_01.json | jq -s '[.[]|.[]|select(.eligible==true) | .overdue as $ov | . += { "ov":$ov|not } ]| sort_by(.ov,.sched_time,.pgid) | [.]  '|  head -20

#cat /tmp/ds_01.json | jq -s '[.[]|.[]|select(.eligible==true) | .overdue as $ov | .level as $lvl | . += { "ov":$ov|not, "lvl":$lvl } ]| sort_by(.ov,.sched_time,.pgid,.lvl) | [.]  '|  head -20

cat /tmp/ds_01.json | jq -s '[.[]|.[]|select(.eligible==true) | .overdue as $ov | .level as $lvl |
 . += { "ov":$ov|not, "lvl":$lvl } ]| sort_by(.ov,.sched_time,.pgid,.lvl) | [.]  '|  head -20

# use 'eligible' as just one more sort criteria
echo "========================== --- "
cat /tmp/ds_01.json | jq -s '[.[]|.[]| .eligible as $ripe | .overdue as $ov | .level as $lvl |
 . += { "not_ripe":$ripe|not, "not_ov":$ov|not, "lvl":$lvl } ] | 
 sort_by(.not_ripe, .not_ov,.sched_time,.pgid,.lvl) | [.]  '|  head -100

# now - make the sort an irregular one: if comparing the the targets of one PG, level tramps sched time



# Termination


In [ ]:
%%bash
echo '-------->' $CEPH_JTEST_ROOT
cd $CEPH_JTEST_ROOT
if [ -z $CEPH_JTEST_ROOT ]; then
    echo "CEPH_JTEST_ROOT is not set"
    [[ -f /tmp/jpath ]] && jp=`cat /tmp/jpath` || jp='.'
    echo '-------->' $jp
    cd $jp
    %env CEPH_JTEST_ROOT=$jp
fi

pwd

../src/stop.sh
sleep 4
../src/stop.sh
